# Flexible Quick visualization

This notebook will generate 8bit visualizations of data in nd2 files and save them as png for quick visualization using e.g. standard file explorers.

If data is 3D, orthogonal projections will be created. Will create multiple output files for other dimensions in nd2 file (e.g. multi-position, time-point, ...)

By changing the do_color setting, you can either generate separate visualizations per channel or a single RGB composite.

In [ ]:
import nd2
import numpy as np
from skimage.io import imsave
from pathlib import Path
import warnings

from calmutils.color import gray_images_to_rgb_composite
from calmutils.color.color import DEFAULT_COLOR_NAMES
from calmutils.misc.visualization import get_orthogonal_projections_8bit
from projection import normalize_intensity_to_8bit

In [ ]:
in_path = '/Volumes/agl_data/Machine_Learning/Cell_cycle_classification/260311 weihua_wen'

raw_subdir = ''
out_subdir = 'quick_visualization'

# Intensity range for leveling
# Default: 'auto' -> level based on quantiles 
intensity_range = 'auto'
auto_range_quantiles = (0.02, 0.9995)

# Alternatively, specify range directly
# intensity_range = (500, 1500)

# Type of projection, can be min/max/mean
projection_type='max'

# whether to produce color visualization (True) or separate for each channel (False) 
do_color = False

# colors for each channel, the default setting is same as in Fiji
visualization_colors = DEFAULT_COLOR_NAMES
# or pick custom
visualization_colors = ('green', 'red', 'blue')

# prefixes for grouping dimensions in filenames
prefixes={"C": "_ch", "T": "_tp", "P": "_pos"}

In [ ]:
# get all nd2 files in input location
in_path = Path(in_path)
in_files = sorted((in_path / raw_subdir).glob('[!.]*.nd2'))

# show for verification
in_files

In [ ]:
# make outdir
out_dir = in_path / out_subdir
if not out_dir.exists():
    out_dir.mkdir()

# main loop
for in_file in in_files:

    # read image to XArray and load pixel sizes
    with nd2.ND2File(in_file) as reader:
        img_xa = reader.to_xarray()
        pixel_size = reader.voxel_size()[::-1]
    
    # group by non-channel or spatial dimensions
    grouping_dims = [d for d in img_xa.dims if d not in ("CZYX" if do_color else "ZYX")]
    # if no grouping dimensions remain, fake single group
    img_grouped = (
        img_xa.groupby(grouping_dims) if len(grouping_dims) > 0 else [((), img_xa)]
    )
    
    for idx, img_i in img_grouped:
    
        # get image data by channel if we want to do color vis and have multiple
        if do_color and "C" in img_i.dims:
            channel_imgs = [img_ci.values.squeeze() for (_, img_ci) in img_i.groupby("C")]
        # otherwise, work with single img
        else:
            channel_imgs = [img_i.values.squeeze()]
    
        # project & normalize or just normalize if we have 2D
        projected_imgs = []
        for channel_img in channel_imgs:
            if channel_img.ndim > 2:
                projected_imgs.append(
                    get_orthogonal_projections_8bit(
                        channel_img,
                        pixel_size=pixel_size,
                        projection_type=projection_type,
                        intensity_range=intensity_range,
                        auto_range_quantiles=auto_range_quantiles,
                    )
                )
            else:
                projected_imgs.append(
                    normalize_intensity_to_8bit(
                        channel_img,
                        intensity_range=intensity_range,
                        auto_range_quantiles=auto_range_quantiles,
                    )
                )
    
        # color projections if we have multiple channels
        if do_color and "C" in img_i.dims:

            # get integers channel indices (groupby works lexicographically)
            channel_idx = [ci for (ci, _) in img_i.groupby("C")]
            channel_idx_int = [img_xa.get_index('C').get_loc(ci) for ci in channel_idx]

            # re-sort to original channel order
            projected_imgs = np.array(projected_imgs)[np.argsort(channel_idx_int)]
            
            composite_img = (
                gray_images_to_rgb_composite(
                    projected_imgs, color_names=visualization_colors
                )
                * 255
            ).astype(np.uint8)
        else:
            composite_img = projected_imgs[0]
    
    
        ## get out filename

        # treat even single split dimension as list of one
        if len(grouping_dims) == 1:
            idx = [idx]
    
        # get integer indices
        # TODO: make settable in parameters?
        use_indices = True
        min_index_len = 1
        
        if use_indices:
            filename_idx = [img_xa.get_index(d).get_loc(i) for d,i in zip(grouping_dims, idx)]
            filename_idx = [str(i).rjust(min_index_len, '0') for i in filename_idx]
        else:
            filename_idx = idx
    
        # construct out filename
        out_filename = Path(in_file).stem + "".join(prefixes[d] + i for d, i in zip(grouping_dims, filename_idx)) + '.png'
        out_filename = out_dir / out_filename
        
        # save, catch low contrast warning
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', UserWarning)
            imsave(str(out_filename), composite_img)

    print(f'saved visualization(s) for {in_file}')